In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import plotly


import src.data_loader as dl
from src.datasets import LatLonData, XYData, merge, project_latlon_data
from src.preclustering import precluster_latlon_boxes
from src.density_clustering import density_clustering
from src.gmm import gmm
from src.utils import min_enclosing_cap
from src.plotting import generate_folium_map, plot_data

%matplotlib inline
plotly.offline.init_notebook_mode()

## Load data

In [ ]:
filepath = 'example 2 months rot.csv'
# filepath = 'RUS last 3 months.csv'

In [ ]:
df, nans = dl.load_data(filepath)

In [ ]:
# Create dict of ElNots + counts, sorted in descending order

elnots = df['elnot']
elnots, counts = np.unique(elnots, return_counts=True)
elnots = {elnot: int(count) for elnot, count in zip(elnots, counts)}
elnots = dict(sorted(elnots.items(), key=lambda item: item[1], reverse=True))
print(elnots)

### Load elnot subset as `LatLonData` instance

In [ ]:
# elnot = 'S962T'
elnot = 'V130A'
data = df[df['elnot'] == elnot]
latlons, ellipses, timestamps = dl.extract_arrays(data, names=['latlons', 'ellipses', 'time'])

data = LatLonData(latlons, ellipses, 
                  timestamps=timestamps,
                  idx=data.index.astype(int))


## Precluster 1
### Split per bounding boxes connected components

In [ ]:
zones, _ = precluster_latlon_boxes(data, check_overlap=True)

In [ ]:
# Analyze zones
split_zone_lengths = [len(zone) for zone in zones]
zone_radii = [min_enclosing_cap(zone.latlons)[1] for zone in zones]
zone_radii = ', '.join([f"{r:.2f}" for r in zone_radii])

print(f"\nAfter initial split into {len(zones)} zones by bbox overlap:")
print(f"Zone sizes [# points]: {split_zone_lengths}")
print(f"Zone radii [degrees]: {zone_radii}")

## Generate folium maps html files
Too large datasets take some time to generate and might crash your browser when opening.

In [ ]:
merged_data = merge(zones)
# Create dir if not exists: maps
import os
if not os.path.exists('maps'):
    os.makedirs('maps')

m = generate_folium_map(merged_data, mode='bbox', tiles=None)
m.save('maps/bbox.html')

m = generate_folium_map(merged_data, mode='ellipse', tiles=None)
m.save('maps/ellipses.html')

# m = generate_folium_map(zones[0], mode='points')
# m.save('maps/zone_1.html')

## Projection
Project zone to 2D data (`XYData` instance) and plotting\
Note: Plotting ellipses for large datasets takes some time

In [ ]:
large_zone = project_latlon_data(data, verbose=True)

In [ ]:
xy_zones = [project_latlon_data(zone) for zone in zones]
len(xy_zones)

In [ ]:
fig = plot_data(xy_zones[0]) 


In [ ]:
# for zone in xy_zones:
#     if len(zone) < 5:
#         continue
#     fig, ax = plot_data(zone, show_ellipses=True)
#     plt.show()
#     plt.close()
    

In [ ]:
testset = xy_zones[0]
subsets, outliers, peaks = density_clustering(testset, 
                                              min_dist_between_peaks=1000,
                                              sigma_min=0,
                                              cell_size_multiplier=1,
                                              sigma_in_meters=300,
                                              threshold_count_per_km2=120,
                                              blob_min_count_per_km2=1,
                                              blob_patience_in_meters=1000,
                                              verbose=True,
                                              blob_lower_tol=.01,
                                              outliers_subgroup_tol=0,
                                              outliers_alpha=.999)

labeled_testset = XYData.merge(subsets)

In [ ]:
subset = subsets[0]
len(subset), len(outliers), len(testset)

In [ ]:
from src.spectral_clustering import spectral_clustering

for i, subset in enumerate(subsets):
    gaps = spectral_clustering(subset, 10,
                        n_neighbors=200,
                        max_degree_ratio=50,
                        z_step=.01,
                        verbose=False,
                        compute_labels=False, 
                        return_gaps=True)
    print(f"Subset {i} gaps:", gaps)
    gmm(subset)
    fig, ax = plot_data(subset, figsize=(16,12), show_ellipses=True)
    fig.savefig(f'img/cluster_{i}.png')
    plt.close(fig)

In [ ]:
subset = subsets[6]
tps = subset.timestamps # np.datetime64
min_of_day = np.array([ts.astype('datetime64[m]').astype(float) % (24*60) for ts in tps])
labels = subset.labels
# Histogram of hour in day of tps per labels
for label in np.unique(labels):
    plt.hist(min_of_day[labels == label], bins=100, alpha=0.5, label=f'Cluster {label}')
plt.xlabel('Minute of Day')
plt.ylabel('Count')
plt.title('Distribution of Observations by Minute of Day per Cluster')
plt.legend()
plt.show()


In [ ]:
# for i, subset in enumerate(subsets):
#     fig, ax = plot_data(subset, figsize=(16,12), show_ellipses=True)
#     fig.savefig(f'img/cluster_{i}_mahal.png')
#     plt.close(fig)
#     # generate_folium_map(data.get_by_index(subset.idx), 
#     #                     ellipse_num_points=10,
#     #                     ellipse_alpha=0.1).save(f'maps/cluster_{i}.html')

In [ ]:
subset = subsets[13]
import plotly
import plotly.graph_objs as go

plotly.offline.init_notebook_mode()

# Main scatter: points colored by label
scatter_points = go.Scattergl(
    x=subset.x,
    y=subset.y,
    mode='markers',
    marker=dict(
        color=subset.labels,
        colorscale='Viridis',
        size=3,
        line_width=0
    )
)
scatter_centers = go.Scattergl()  # empty

if hasattr(subset, 'mu') and subset.mu is not None:
    scatter_centers = go.Scattergl(
        x=subset.mu[:, 0],
        y=subset.mu[:, 1],
        mode='markers',
        marker=dict(
            color=np.arange(len(subset.mu)),
            colorscale='Viridis',
            size=14,
            line=dict(width=2, color='black')
        )
    )

fig = go.Figure(data=[scatter_points, scatter_centers])
fig.update_layout(
    dragmode='pan',
    yaxis=dict(scaleanchor="x", scaleratio=1),
    title="Clustered points and centers (equal axis scale)",
    xaxis_title="X [meters]",
    yaxis_title="Y [meters]",
    width=900,
    height=900,
    showlegend=False
)
fig.show(config={'scrollZoom': True})


In [ ]:
# dx = target.pos - target.mu[None, 0]
# tps = target.timestamps
# # Sort both by timestamps
# sorted_indices = np.argsort(tps)
# tps = tps[sorted_indices]
# dx = dx[sorted_indices]

# dists_to_mu = np.linalg.norm(dx, axis=1)

# # Compute first degree polynomial fit (linear fit)
# coefficients = np.polyfit(tps.astype(float), dists_to_mu, 1)
# polynomial = np.poly1d(coefficients)
# trendline = polynomial(tps.astype(float))

# # convert polynomial 1st degree to velocity in km/days (current is tps in s, dists in meters)
# velocity_days = coefficients[0] * 3.6 * 24 # km/days
# velocity_days

In [ ]:
# sorted_time = np.sort(tps.astype(float))
# (sorted_time[-1] - sorted_time[0]) / 3600 / 24

In [ ]:
# import plotly
# import plotly.graph_objs as go

# # Configure Plotly to be rendered inline in the notebook.
# plotly.offline.init_notebook_mode()

# # Configure the trace.
# trace = go.Scatter3d(
#     x=tps,
#     y=dx[:,0],
#     z=dx[:,1],
#     mode='markers',
#     marker=dict(size=5, color=tps.astype(float), colorscale='Viridis', opacity=0.8),
#     text=[f'Time: {tp}<br>dx: {d[0]:.0f} m<br>dy: {d[1]:.0f} m<br>Distance to GMM center: {dist:.0f} m' 
#           for tp, d, dist in zip(tps, dx, dists_to_mu)]
# )

# data = [trace]

# layout = go.Layout(
#     title='3D Scatter Plot of Distances to GMM Center Over Time',
#     scene=dict(
#         xaxis_title='Timestamp',
#         yaxis_title='dx (meters)',
#         zaxis_title='dy (meters)'
#     ),
#     margin=dict(l=0, r=0, b=0, t=30)
# )

# # Create the figure.
# plot_figure = go.Figure(data=data, layout=layout)

# # Render the plot.
# plotly.offline.iplot(plot_figure)

In [ ]:
to_plot = subset
folium_data = data.get_by_index(to_plot.idx)

In [ ]:
# 1. Get the indices that would sort A.idx
sorter = np.argsort(to_plot.idx)

# 2. Find the positions of B.idx elements within the sorted A.idx
#    This gives us an array of indices that maps B.idx to the sorted A.idx
positions = np.searchsorted(to_plot.idx, folium_data.idx, sorter=sorter)

# 3. Use the sorter to get the final indices into the original A.labels
final_indices = sorter[positions]

# 4. Assign the correctly ordered labels from A to B
folium_data.labels = to_plot.labels[final_indices]

In [ ]:
m = generate_folium_map(folium_data, mode='points', ellipse_num_points=12)
m.save('maps/labeled_set_points.html')

## Second split
Compute weighted graph based on pairwise mahalanobis distances and split into connected components.

In [ ]:
import cProfile
import pstats

In [ ]:
pr = cProfile.Profile()
pr.enable()
df, nans = dl.load_data(filepath)
pr.disable()

In [ ]:
stats = pstats.Stats(pr)
stats.sort_stats('cumulative')
stats.print_stats(20)

In [ ]:
len(subset)